In [ ]:
# All Import Statements Defined Here
# Note: Do not add to this list.
# All the dependencies you need, can be installed by running .
# ----------------
 
import sys
assert sys.version_info[0]==3
assert sys.version_info[1] >= 5
 
 
 
 
 
from gensim.models import KeyedVectors
from gensim.test.utils import datapath
import pprint
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = [10, 5]
import nltk
nltk.download('reuters')
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')
from nltk.corpus import reuters
import numpy as np
import random
import scipy as sp
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import PCA
 
START_TOKEN = '<START>'
END_TOKEN = '<END>'
 
np.random.seed(0)
random.seed(0)
# ----------------

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [ ]:
def load_word2vec():
    """ Load Word2Vec Vectors
        Return:
            wv_from_bin: All 3 million embeddings, each lengh 300
    """
    import gensim.downloader as api
    wv_from_bin = api.load("word2vec-google-news-300")
    # wv_from_bin = api.load("glove-wiki-gigaword-200")
    vocab = list(wv_from_bin.vocab.keys())
    print("Loaded vocab size %i" % len(vocab))
    return wv_from_bin
  
# -----------------------------------
# Run Cell to Load Word Vectors
# Note: This may take several minutes
# -----------------------------------
wv_from_bin = load_word2vec()

Loaded vocab size 3000000


# Extracting All Adjectives

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:

# dog_breeds_file = 'drive/MyDrive/DS4A Team 73/DS4A_projectdata/NLP Data/StandardDogBreeds.json'
# try:
#     dog_breeds_json = json.load(open(dog_breeds_file))
# except FileNotFoundError:
#     print('WARNING: Database file not found.')

# list_of_subbreeds = [(subbreed + ' ' + breed) for breed, list_of_subbreeds in dog_breeds_json.items() for subbreed in list_of_subbreeds ]
# list_of_breeds = [breed for breed, list_of_subbreeds in dog_breeds_json.items()]
# full_list_of_dogs = list_of_subbreeds + list_of_breeds

In [ ]:
# full_list_of_dogs
# dog_name = 'pitbull'
# reddit_data_folder = 'drive/MyDrive/DS4A Team 73/DS4A_projectdata/NLP Data/Reddit Posts/'
# file_name = dog_name + '.json'
 
# dog_comments_json = json.load(open(reddit_data_folder + file_name))
 
import json
import nltk
import nltk.corpus
import matplotlib.pyplot as plt
import math
nltk.download("punkt")
nltk.download('averaged_perceptron_tagger')
 
 
 
labrador_comments_json = json.load(open("labrador.json"))
pitbull_comments_json = json.load(open("pitbull.json"))
dog_comments = pitbull_comments_json + labrador_comments_json
 
 
token_sentences = [nltk.word_tokenize(sent) for sent in dog_comments]
tagged_sentences = [nltk.pos_tag(sent) for sent in token_sentences] 
filtered_text = [dog for l in tagged_sentences for (dog, POS) in l if (POS == "JJ") and len(dog) > 2]
 
 

def unit_vector(vector):
    """ Returns the unit vector of the vector.  """
    return vector / np.linalg.norm(vector)
def angle_between(v1, v2):
    """ Returns the angle in radians between vectors 'v1' and 'v2'::
            >>> angle_between((1, 0, 0), (0, 1, 0))
            1.5707963267948966
            >>> angle_between((1, 0, 0), (1, 0, 0))
            0.0
            >>> angle_between((1, 0, 0), (-1, 0, 0))
            3.141592653589793
    """
    v1_u = unit_vector(v1)
    v2_u = unit_vector(v2)
    return np.arccos(np.clip(np.dot(v1_u, v2_u), -1.0, 1.0))
# The more positive the value is the more the vector aligns with the goodness vector, if it is negative, it is opposite to good
def cosine(word, goodness_unit_vector=goodness_unit_vector):
  cosine = math.cos(angle_between(wv_from_bin.word_vec(word), goodness_unit_vector))
  return cosine

goodness_unit_vector = unit_vector(wv_from_bin.word_vec("good") - wv_from_bin.word_vec("bad"))

def good_words (words , top =5 ):
  real_cosine = {}
  for word in words :
    try:
      real_cosine[word] = cosine(word) 
    except:
      real_cosine[word] = 0

  sorted_words= sorted(real_cosine,key=real_cosine.get,reverse=True)[:top]
  return sorted_words

def bad_words(words , top =5):
  real_cosine = {}
  for word in words :
    try:
      real_cosine[word] = cosine(word) 
    except:
      real_cosine[word] = 0

  sorted_words= sorted(real_cosine,key=real_cosine.get,reverse=True)[-top:]
  return sorted_words


good_words(filtered_text,10)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


NameError: ignored

['horrible',
 'crippling',
 'toxic',
 'unpleasant',
 'horrid',
 'hideous',
 'sour',
 'ugly',
 'nasty',
 'bad']

In [ ]:
from datetime import datetime
from bs4 import BeautifulSoup
import argparse
import requests
import json
import re
 
SITE_URL = 'https://old.reddit.com/'
REQUEST_AGENT = 'Mozilla/5.0 Chrome/47.0.2526.106 Safari/537.36'
 
def createSoup(url):
    return BeautifulSoup(requests.get(url, headers={'User-Agent':REQUEST_AGENT}).text, 'lxml')
 
def getSearchResults(searchUrl):
    posts = []
    while True:
        resultPage = createSoup(searchUrl)
        posts += resultPage.findAll('div', {'class':'search-result-link'})
        footer = resultPage.findAll('a', {'rel':'nofollow next'})
        if footer:
            searchUrl = footer[-1]['href']
        else:
            return posts
 
def parsePosts(posts, product, keyword):
    for post in posts:
        time = post.find('time')['datetime']
        date = datetime.strptime(time[:19], '%Y-%m-%dT%H:%M:%S')
        title = post.find('a', {'class':'search-title'}).text
        score = post.find('span', {'class':'search-score'}).text
        score = int(re.match(r'[+-]?\d+', score).group(0))
        author = post.find('a', {'class':'author'}).text
        subreddit = post.find('a', {'class':'search-subreddit-link'}).text
        commentsTag = post.find('a', {'class':'search-comments'})
        url = commentsTag['href']
        numComments = int(re.match(r'\d+', commentsTag.text).group(0))
        product[keyword].append({'title':title, 'url':url, 'date':str(date),
                                 'score':score, 'author':author, 'subreddit':subreddit})
    return product
 
 
def parseComments(commentsUrl):
    commentTree = {}
    commentsPage = createSoup(commentsUrl)
    commentsDiv = commentsPage.find('div', {'class':'sitetable nestedlisting'})
    comments = commentsDiv.findAll('div', {'data-type':'comment'})
    for comment in comments:
        numReplies = int(comment['data-replies'])
        tagline = comment.find('p', {'class':'tagline'})
        author = tagline.find('a', {'class':'author'})
        author = "[deleted]" if author == None else author.text
        date = tagline.find('time')['datetime']
        date = datetime.strptime(date[:19], '%Y-%m-%dT%H:%M:%S')
        commentId = comment.find('p', {'class':'parent'}).find('a')['name']
        content = comment.find('div', {'class':'md'}).text.replace('\n','')
        score = comment.find('span', {'class':'score unvoted'})
        score = 0 if score == None else int(re.match(r'[+-]?\d+', score.text).group(0))
        parent = comment.find('a', {'data-event-action':'parent'})
        parentId = parent['href'][1:] if parent != None else ''
        parentId = '' if parentId == commentId else parentId
        commentTree[commentId] = {'author':author, 'reply-to':parentId, 'text':content,
                                  'score':score, 'num-replies':numReplies, 'date':str(date)}
    return commentTree
 
if __name__ == '__main__':
    keyword = "pitbull"
    searchUrl = SITE_URL + 'search?q="' + keyword + '"'
 
    try:
        product = json.load(open('product.json'))
    except FileNotFoundError:
        print('WARNING: Database file not found. Creating a new one...')
        product = {}
    print('Search URL:', searchUrl)
    posts = getSearchResults(searchUrl)
    print('Started scraping', len(posts), 'posts.')
    keyword = keyword.replace(' ', '-')
    product[keyword] = []
    product = parsePosts(posts, product, keyword)
    with open('product.json', 'w', encoding='utf-8') as f:
        json.dump(product, f, indent=4, ensure_ascii=False)

Search URL: https://old.reddit.com/search?q="pitbull"
Started scraping 247 posts.
